# Judge-Driven Training Loop

Close the loop: **train → judge → curate → SFT → retrain**. Three runs
of GSPO, each one trained on the high-scoring outputs of the previous
run as scored by an LLM judge. The cheapest realistic version of the
RLHF loop short of human labels.

**Runtime:** ~90 min on Colab A100. **Cost:** ~$0.80.

**Inspired by** whitepaper §6.2 (continual learning) and
`grade_and_curate_demo.ipynb` (chat → grade → curate), but with the
feedback loop closed end-to-end instead of being two separate notebooks.


## 1. Pin + install


In [ ]:
import os
import subprocess

PINNED_COMMIT = 'f8e498b'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training]'
%pip install --quiet accelerate
print('Installed.')


In [ ]:
from stateset_agents.utils.reproducibility import set_all_seeds

SEED = 42
state = set_all_seeds(SEED)
print('Seeds applied:', state.to_dict())


## 2. Pick a domain — we use the bundled support corpus


In [ ]:
from stateset_agents.data import load_support_scenarios, SupportRewardComposite

scenarios = load_support_scenarios()
TRAIN, EVAL = scenarios[:16], scenarios[16:]
RUBRIC = SupportRewardComposite()
print(f'{len(TRAIN)} train, {len(EVAL)} eval scenarios.')


## 3. Load a small LLM judge

Qwen2.5-1.5B-Instruct is the same judge used in the §11.7 protocol — small
enough to fit alongside a 0.5B trainee on an A100, big enough to be
paraphrase-tolerant. *Note:* the judge sees only `(query, intent, response)`,
not the ground-truth rubric — that's the whole point.


In [ ]:
import torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer

JUDGE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
judge_tok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
judge_lm = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL, torch_dtype='bfloat16', attn_implementation='sdpa').cuda().eval()

JUDGE_PROMPT = ("You are evaluating a customer service agent's response. "
                'Rate 0-10. Customer: {q}  Intent: {i}  Agent: {r}  Score:')

@torch.no_grad()
def judge_score(q, i, r):
    prompt = JUDGE_PROMPT.format(q=q, i=i, r=r)
    inputs = judge_tok(prompt, return_tensors='pt').to('cuda')
    out = judge_lm.generate(**inputs, max_new_tokens=3, do_sample=False)
    text = judge_tok.decode(out[0, inputs.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r'\d+', text)
    return min(10, int(m.group())) / 10.0 if m else 0.0

print('Judge loaded. Smoke test:',
      judge_score('I need a refund', 'refund', 'I can process that right away.'))


## 4. The training step (one iteration of the loop)


In [ ]:
from stateset_agents import MultiTurnAgent
from stateset_agents.core.agent_config import AgentConfig
from stateset_agents.core import ConversationEnvironment
from stateset_agents.training import GSPOConfig, train_with_gspo
from stateset_agents.data import make_support_scenarios

MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

def prompt_for(s):
    return (
        'You are a helpful customer support agent. Respond warmly, '
        f"address the concern, confirm the next step.\n\nUser: {s.user_query}\n\nAgent:"
    )

async def train_one_iteration(iteration_idx: int):
    output_dir = f'/content/gspo_judge_iter{iteration_idx}'
    config = GSPOConfig(
        model_name=MODEL, output_dir=output_dir, report_to='none',
        num_generations=4, clip_range_left=3e-4, clip_range_right=4e-4,
        learning_rate=5e-6, max_prompt_length=512, max_completion_length=320,
        use_lora=True, lora_r=16, lora_alpha=32,
        num_epochs=1, warmup_ratio=0.1,
        use_reference_model=True, beta=0.05,
    )
    agent = MultiTurnAgent(AgentConfig(model_name=MODEL, torch_dtype='bfloat16', attn_implementation='sdpa'))
    env = ConversationEnvironment(scenarios=make_support_scenarios(TRAIN), reward_fn=RUBRIC, max_turns=2)
    queries = [{'prompt': prompt_for(s), 'context': s.to_scenario()} for s in TRAIN]
    await train_with_gspo(config=config, agent=agent, environment=env,
                          reward_model=env.reward_fn, train_queries=queries)
    return agent, output_dir

print('Helpers defined.')


## 5. Two iterations + judge-scored eval


In [ ]:
import statistics
from datetime import datetime, timezone
from pathlib import Path
import json

async def judge_eval(agent):
    scores = []
    for s in EVAL:
        response = await agent.generate_response(prompt_for(s))
        scores.append(judge_score(s.user_query, s.intent, response))
    return statistics.mean(scores), scores

history = []
for it in range(2):
    print(f'\n=== Iteration {it+1} ===')
    agent, out_dir = await train_one_iteration(it)
    mean, scores = await judge_eval(agent)
    print(f'Iter {it+1} judge mean: {mean:.3f}  scores: {[f"{s:.2f}" for s in scores]}')
    history.append({'iteration': it+1, 'judge_mean': mean, 'judge_scores': scores, 'output_dir': out_dir})
    del agent
    torch.cuda.empty_cache()

out = Path('/content/judge_loop_history.json')
out.write_text(json.dumps({
    'model': MODEL, 'judge': JUDGE_MODEL, 'seed': SEED,
    'pinned_commit': PINNED_COMMIT,
    'iterations': history,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
}, indent=2))
print(f'\nWrote {out}')


## 6. Read the trajectory

If `iter 2 > iter 1`, the judge-scored loop converged in the right direction.
If they're flat, your judge isn't differentiating — fall back to a stronger
rubric or scale the judge (Qwen2.5-7B-Instruct). If iter 2 < iter 1, you've
likely hit reward hacking — the agent is satisfying the judge in ways that
drift from the underlying rubric. Run the rubric eval alongside as a sanity check.


In [ ]:
for h in history:
    print(f"iter {h['iteration']}: judge={h['judge_mean']:.3f}")


## 7. Caveats

- **A 1.5B judge is the floor.** Use it for smoke tests; publication numbers
  should use a 7B+ judge or human labels.
- **Reward hacking is real.** Always co-eval against the rubric to catch it.
- **Don't loop forever.** Two-three iterations is plenty for a smoke run;
  longer loops need explicit early-stopping criteria.
